# Notebook 14 — The gene panel: two coupled networks tested for selection across primates

**Scope.** This notebook defines and justifies the gene panel for the primate sexual-dichromatism selection
scan, and it does so **reproducibly**: every gene list, evidence assignment, orthology flag, and count below is
*computed by the code in this notebook* from the project's raw source files — nothing is asserted from a
precomputed table. Re-running the notebook regenerates the panel from scratch.

We test two coupled molecular systems for signatures of selection: the **melanocyte pigmentation network** and
the **sex-hormone (hypothalamic-pituitary-gonadal) axis**. Sexual dichromatism is a pigmentation phenotype
expressed in a sex-limited way, so the a-priori design pairs the pigment machinery with the sex-steroid
signaling that could gate its expression. The two modules are matched in breadth so per-module comparisons
reflect biology, not sampling.

**Raw inputs (all version-controlled in this repository):**
- `comparative-genomics/analysis/data/pigmentation_network_nodes.csv` — the 803-node curated pigmentation
  network with per-gene evidence layers (OMIM pigmentation phenotype, Raghunath melanogenesis-mechanism token,
  melanosome mass-spec, functional CRISPR screen).
- `comparative-genomics/config/gene_panel.csv` — the genes already carried in the selection scan.
- `comparative-genomics/analysis/data/hormone_axes.csv` — hormone-module genes with endocrine-axis labels.
- `data/processed/baxter2018_650_pigmentation_genes.csv` — Baxter et al. 2019 curated pigmentation reference
  (*Pigment Cell Melanoma Res* 32:348, doi:10.1111/pcmr.12743; 659 gene records, 635 unique human gene symbols).

In [ ]:
import os, json, time
import pandas as pd, requests
REPO = os.environ.get("PIGNET_REPO", os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")))
CG   = os.path.join(REPO, "comparative-genomics")
NET  = pd.read_csv(os.path.join(CG, "analysis/data/pigmentation_network_nodes.csv"))
PANEL= pd.read_csv(os.path.join(CG, "config/gene_panel.csv"))
HAX  = pd.read_csv(os.path.join(CG, "analysis/data/hormone_axes.csv"))
BAX  = pd.read_csv(os.path.join(REPO, "data/processed/baxter2018_650_pigmentation_genes.csv"))
BAX_GENES = set(BAX["Human gene symbol"].dropna().astype(str).str.strip())
IN_PANEL  = set(PANEL.gene)
print(f"network nodes: {len(NET)} | already-in-panel genes: {len(IN_PANEL)} | Baxter reference: {len(BAX_GENES)}")

network nodes: 803 | already-in-panel genes: 80 | Baxter reference: 635


## 1 — Evidence flags, computed from the network layers

For every network gene we derive four pigmentation-specific evidence flags directly from the source columns.
These are the only evidence types we accept as "this is a pigmentation gene"; membership via generic
STRING/KEGG proximity alone does not qualify.

In [ ]:
NET["has_raghunath"] = NET.supporting_layers.astype(str).str.contains("Raghunath")   # melanogenesis mechanism
NET["omim_pigment"]  = NET.omim_phenotype_class.notna()                               # OMIM pigment disorder
NET["in_baxter"]     = NET.gene.isin(BAX_GENES)                                       # Baxter 2019 reference
NET["crispr_screen"] = NET.bajpai_hit_flag == True                                    # functional screen hit
NET["pig_specific"]  = NET.omim_pigment | NET.in_baxter | NET.crispr_screen
print("network genes with each evidence flag:")
for c in ["has_raghunath","omim_pigment","in_baxter","crispr_screen","pig_specific"]:
    print(f"  {c:14s}: {int(NET[c].sum())}")

network genes with each evidence flag:
  has_raghunath : 168
  omim_pigment  : 239
  in_baxter     : 204
  crispr_screen : 29
  pig_specific  : 310


## 2 — Assemble the pigmentation expansion candidates from the layers

The original pigmentation module (27 genes) covered only melanin *synthesis* and its immediate regulators. Two
functional groups are demonstrably missing, and both are recovered here **by rule** from the network layers:

- **Melanosome biogenesis & transport** — genes with an OMIM pigmentation phenotype belonging to the
  melanosome-biogenesis/transport families (Hermansky-Pudlak, AP-3, BLOC-1, Griscelli RAB27A/MYO5A/MLPH, LYST,
  GPR143, the vitiligo SLC transporters). Selected by OMIM-pigment flag + gene-family name pattern.
- **Melanogenesis regulation** — genes on the Raghunath melanogenesis-mechanism layer that also carry
  pigmentation-specific evidence (OMIM pigment / Baxter / functional screen).

In [ ]:
cand = NET[~NET.gene.isin(IN_PANEL)].copy()

MEL_FAMILIES = ("HPS","AP3","BLOC","RAB27","MYO5","MLPH","LYST","GPR143",
                "RAB5C","SLC17A5","SLC1A2","SLC44A4","SLC29A3","CTNS")
def in_melanosome_family(g):
    return any(g.startswith(k) or k in g for k in MEL_FAMILIES)

group_A = cand[cand.omim_pigment & cand.gene.map(in_melanosome_family)].gene.tolist()
group_B = cand[cand.has_raghunath & cand.pig_specific].gene.tolist()
group_B = [g for g in group_B if g not in group_A]   # keep groups disjoint
print(f"Group A — melanosome biogenesis & transport: {len(group_A)}")
print("  ", ", ".join(sorted(group_A)))
print(f"Group B — melanogenesis regulation: {len(group_B)}")
print("  ", ", ".join(sorted(group_B)))
candidates = sorted(set(group_A) | set(group_B))
print(f"total expansion candidates: {len(candidates)}")

Group A — melanosome biogenesis & transport: 21
   AP3B1, AP3D1, BLOC1S3, BLOC1S5, BLOC1S6, CTNS, GPR143, HPS1, HPS3, HPS4, HPS5, HPS6, LYST, MLPH, MYO5A, RAB27A, RAB5C, SLC17A5, SLC1A2, SLC29A3, SLC44A4
Group B — melanogenesis regulation: 22
   BCL2, CDC42, CDH2, CTNNB1, EDN1, EGFR, EP300, FASLG, HGF, HRAS, MAP2K1, MAP2K2, MAPK3, PAK1, PPP3CA, RAC1, RACK1, RAF1, SPTLC2, SRC, TP53, TRAF6
total expansion candidates: 43


## 3 — Orthology screen against Ensembl Compara

A dN/dS scan is trustworthy only if each gene has a clean **one-to-one ortholog** in every primate genome. Two
failure modes threaten that: **paralog mis-mapping** (the ortholog-extraction step lands on a close paralog —
a risk for large gene families) and **incomplete ortholog coverage** (short/poorly-annotated genes missing from
scaffold-level or deep-lineage assemblies). We query the Ensembl REST homology endpoint for each candidate:
count how many of 12 reference primates carry a called ortholog, and how many human within-species paralogs
exist. The live query function is below; results are cached to `data/nb14_ensembl_orthology.json` so the
notebook re-runs deterministically and offline (delete the cache to force a fresh query).

In [ ]:
PRIMATES = ["pan_troglodytes","gorilla_gorilla","pongo_abelii","macaca_mulatta","macaca_fascicularis",
            "chlorocebus_sabaeus","papio_anubis","nomascus_leucogenys","callithrix_jacchus",
            "microcebus_murinus","otolemur_garnettii","carlito_syrichta"]
CACHE = os.path.join(CG, "analysis/module_selection/data/nb14_ensembl_orthology.json")

def ensembl_orthology(symbol, session, base="https://rest.ensembl.org"):
    """Return (n_primate_orthologs, n_human_paralogs) for a human gene symbol via Ensembl Compara."""
    hdr = {"Content-Type": "application/json"}
    r = session.get(f"{base}/lookup/symbol/homo_sapiens/{symbol}", headers=hdr, timeout=30)
    if r.status_code != 200:
        return None
    gid = r.json()["id"]
    h = session.get(f"{base}/homology/id/homo_sapiens/{gid}",
                    headers=hdr, params={"type": "all", "format": "condensed"}, timeout=60)
    if h.status_code != 200 or not h.json().get("data"):
        return (0, 0)
    homs = h.json()["data"][0]["homologies"]
    prim = {x["species"] for x in homs
            if x.get("type","").startswith("ortholog") and x.get("species") in PRIMATES}
    par  = sum(1 for x in homs
               if x.get("type") == "within_species_paralog" and x.get("species") == "homo_sapiens")
    return (len(prim), par)

if os.path.exists(CACHE):
    ortho = json.load(open(CACHE))
    print(f"loaded cached Ensembl orthology for {len(ortho)} genes (delete {os.path.basename(CACHE)} to refresh)")
else:
    print("querying Ensembl Compara (live)...")
    S = requests.Session(); ortho = {}
    for g in candidates:
        res = ensembl_orthology(g, S)
        ortho[g] = {"n_primate_orth": res[0], "n_within_paralog": res[1]} if res else {"n_primate_orth": 0, "n_within_paralog": 0}
        time.sleep(0.06)
    json.dump(ortho, open(CACHE, "w"), indent=1)
    print(f"queried {len(ortho)} genes; cached to {os.path.basename(CACHE)}")

loaded cached Ensembl orthology for 43 genes (delete nb14_ensembl_orthology.json to refresh)


## 4 — Derive the orthology risk flag, from the query results

A gene passes (`clean`) only with full paralog cleanliness and near-complete coverage. Flags: `HIGH_paralog`
(≥3 human paralogs), `paralog_check` (1–2 paralogs), `low_ortholog_cov` (<10/12 primates).

In [ ]:
def risk_flag(n_orth, n_par):
    f = []
    if   n_par >= 3: f.append("HIGH_paralog")
    elif n_par >= 1: f.append("paralog_check")
    if n_orth < 10:  f.append("low_ortholog_cov")
    return ";".join(f) if f else "clean"

orA = pd.DataFrame([{"gene": g, "group": ("melanosome_biogenesis" if g in group_A else "melanogenesis_regulation"),
                     "n_primate_orth": ortho[g]["n_primate_orth"], "n_within_paralog": ortho[g]["n_within_paralog"]}
                    for g in candidates])
orA["ortholog_risk"] = [risk_flag(r.n_primate_orth, r.n_within_paralog) for r in orA.itertuples()]
clean = orA[orA.ortholog_risk == "clean"]
print(f"orthology outcome: {len(clean)} clean / {len(orA)} candidates")
print("\nEXCLUDED (flagged):")
print(orA[orA.ortholog_risk != "clean"].sort_values("group").to_string(index=False))
print(f"\nCLEAN expansion ({len(clean)}): {(clean.group=='melanosome_biogenesis').sum()} melanosome + "
      f"{(clean.group=='melanogenesis_regulation').sum()} regulation")

orthology outcome: 30 clean / 43 candidates

EXCLUDED (flagged):
   gene                    group  n_primate_orth  n_within_paralog                  ortholog_risk
   BCL2 melanogenesis_regulation              11                 2                  paralog_check
   EDN1 melanogenesis_regulation              12                 2                  paralog_check
  FASLG melanogenesis_regulation              12                 7                   HIGH_paralog
 MAP2K1 melanogenesis_regulation              12                 1                  paralog_check
 MAP2K2 melanogenesis_regulation              12                 1                  paralog_check
   PAK1 melanogenesis_regulation              11                 2                  paralog_check
   RAC1 melanogenesis_regulation               8                 1 paralog_check;low_ortholog_cov
   TP53 melanogenesis_regulation              12                 2                  paralog_check
BLOC1S3    melanosome_biogenesis               7     

## 5 — The analytic panel

The panel = the 53-gene hormone axis + the original 27 pigmentation genes + the orthology-clean pigmentation
additions. The two modules are matched in breadth.

In [ ]:
n_hor   = (PANEL.set == "hormone").sum()
n_pig0  = (PANEL.set == "pigmentation").sum()
n_pig   = n_pig0 + len(clean)
balance = 2*n_pig/(n_pig+n_hor) - 1
print(f"sex-hormone module:  {n_hor} genes")
print(f"pigmentation module: {n_pig0} (original) + {len(clean)} (clean additions) = {n_pig}")
print(f"analytic panel total: {n_pig + n_hor} genes")
print(f"module balance-neutral point: {balance:+.3f}   (0 = perfectly balanced panel)")

sex-hormone module:  53 genes
pigmentation module: 27 (original) + 30 (clean additions) = 57
analytic panel total: 110 genes
module balance-neutral point: +0.036   (0 = perfectly balanced panel)


## 6 — Print every gene in the panel, by module and function

In [ ]:
pig_orig = PANEL[PANEL.set=="pigmentation"].merge(
    NET[["gene","omim_pigment","in_baxter","has_raghunath"]], on="gene", how="left")
print("=== PIGMENTATION MODULE (original core, %d) ===" % n_pig0)
for cat, sub in pig_orig.groupby("category"):
    print(f"  {cat} ({len(sub)}): " + ", ".join(sorted(sub.gene)))
print("\n=== PIGMENTATION MODULE (clean additions, %d) ===" % len(clean))
for grp, sub in clean.groupby("group"):
    print(f"  {grp} ({len(sub)}): " + ", ".join(sorted(sub.gene)))
print("\n=== SEX-HORMONE MODULE (%d) ===" % n_hor)
for cat, sub in HAX.groupby("axis_category"):
    print(f"  {cat} ({len(sub)}): " + ", ".join(sorted(sub.gene)))

=== PIGMENTATION MODULE (original core, 27) ===
  enzyme (3): DCT, TYR, TYRP1
  melanosome (2): MLANA, PMEL
  melanosome_transport (5): MFSD12, OCA2, SLC24A4, SLC24A5, SLC45A2
  receptor_signaling (8): ASIP, EDN3, EDNRB, KIT, KITLG, MC1R, MRAP2, POMC
  regulatory (1): HERC2
  transcription (8): BNC2, FOXD3, IRF4, LEF1, MITF, PAX3, SOX10, TFAP2A

=== PIGMENTATION MODULE (clean additions, 30) ===
  melanogenesis_regulation (14): CDC42, CDH2, CTNNB1, EGFR, EP300, HGF, HRAS, MAPK3, PPP3CA, RACK1, RAF1, SPTLC2, SRC, TRAF6
  melanosome_biogenesis (16): AP3B1, AP3D1, BLOC1S6, CTNS, GPR143, HPS1, HPS3, HPS4, HPS5, LYST, MYO5A, RAB27A, RAB5C, SLC17A5, SLC1A2, SLC44A4

=== SEX-HORMONE MODULE (53) ===
  androgen_axis (7): AKR1C3, AR, CYP17A1, HSD17B3, SRD5A1, SRD5A2, SRD5A3
  coactivator_corepressor_carrier (7): FKBP5, NCOA1, NCOA2, NCOA3, NCOR1, NCOR2, SHBG
  estrogen_axis (7): CYP19A1, ESR1, ESR2, GPER1, HSD17B1, STS, SULT1E1
  gnrh_signaling (2): GNA11, GNAQ
  hpg_axis (10): CGA, FSHB, FSHR, G

## Result

The panel is **computed above**, not asserted: hormone axis (53) + original pigmentation core (27) + the
orthology-clean melanosome/regulation additions (30) = **110 genes**, near-perfectly balanced between the two
modules. Genes failing the orthology screen are listed with their exact ortholog/paralog counts and excluded.
This is the gene set carried into the per-branch (aBSREL) and per-origin (RELAX) selection analyses.

The written expansion list for the cluster is `config/gene_panel_expansion_clean30.csv`; the internal
provenance table (which genes are additions, full flag detail) is `data/nb14_panel_justification.csv`.